# DRMD + interacting-ncdm parameter-space validation

This notebook runs structured one-parameter scans comparing my CLASS implementation (`classy_NEDE`) against Tobias’s implementation.

## Goal
The purpose is to test agreement across physically relevant regions of parameter space, rather than only at a single benchmark point.

## Strategy
For each scan:
- fix a benchmark cosmology and benchmark DRMD / interacting-ncdm model
- vary one parameter at a time
- compute background quantities and matter power spectrum comparisons
- extract summary metrics such as:
  - worst mismatch
  - best mismatch
  - worst-k location
  - key derived quantities

## Output
Each scan should produce:
- a detailed results table
- a summary table
- optional plots of mismatch vs scanned parameter

The notebook relies on a reusable Python backend module that contains the scan logic.

In [8]:
# Cell 2 — Import standard libraries and load the reusable scan backend from file path

import importlib.util
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

backend_path = Path("/Users/magnusuggerhoj/Desktop/Speciale/CLASS_NEDE/DRMD-CLASS/Magnus_files/DRMD/drmd_scan_backend.py")

if not backend_path.exists():
    raise FileNotFoundError(f"Backend file not found: {backend_path}")

module_name = "drmd_scan_backend"

spec = importlib.util.spec_from_file_location(module_name, backend_path)
backend = importlib.util.module_from_spec(spec)
sys.modules[module_name] = backend
spec.loader.exec_module(backend)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

In [9]:
# Cell 3 — Define the benchmark model and the parameter scan ranges

benchmark = backend.get_benchmark_params()

scan_ranges = {
    "delta_Neff_drmd": [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
    "f_idm_drmd": [0.0, 0.01, 0.03, 0.05, 0.08, 0.12, 0.18, 0.24, 0.30],
    "z_stop": [1e3, 3e3, 1e4, 3e4, 5e4, 1e5],
    "G_over_aH_drmd_ini": [1e-3, 1e0, 1e3, 1e6, 1e9],
    "deg_ncdm_interacting": [0.2, 0.6, 1.0, 1.4, 1.8, 2.2, 2.6, 3.0, 3.5, 4.0],
    "G_eff_ncdm_interacting": [0.0, 1e-4, 1e-3, 1e-2, 1e-1],
    "m_ncdm_interacting": [1e-3, 3e-3, 1e-2, 3e-2, 6e-2, 1e-1],
}

k_values_1Mpc = [1e-2, 5e-2, 2e-1, 1.0]

benchmark

{'H0': 67.32,
 'omega_b': 0.02238,
 'omega_cdm': 0.1201,
 'A_s': 2.101e-09,
 'n_s': 0.966,
 'tau_reio': 0.0543,
 'k_pivot': 0.05,
 'T_cmb': 2.7255,
 'Omega_k': 0.0,
 'deg_ncdm_interacting': 1.0,
 'm_ncdm_interacting': 0.06,
 'T_ncdm_interacting': 0.71611,
 'G_eff_ncdm_interacting': 0.1,
 'delta_Neff_drmd': 0.8,
 'f_idm_drmd': 0.03,
 'z_stop': 50000.0,
 'G_over_aH_drmd_ini': 1000000000.0}

In [10]:
# Cell 4 — Define the selected physically motivated k values used for comparison

k_values_1Mpc = [1e-2, 5e-2, 2e-1, 1.0]
k_values_1Mpc

[0.01, 0.05, 0.2, 1.0]

In [15]:
# Cell 4 — Smoke test the backend on a tiny delta_Neff_drmd scan

df_test = backend.run_parameter_scan(
    parameter_name="delta_Neff_drmd",
    scan_values=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    benchmark=benchmark,
    k_values_1Mpc=k_values_1Mpc,
    verbose=False,
)

backend.display_scan_table(df_test, parameter_name="delta_Neff_drmd")

DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01
DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01
DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01
DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01
DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01
DEBUG split(Geff_final): Nstd=0 Nint=1 Ntot=1  Geff_species=1.000e-01


,scan_family,parameter_name,parameter_value,worst_selected_k_1Mpc,worst_abs_mine_over_tobias_minus1_pct,worst_mine_over_tobias_minus1_pct,best_selected_k_1Mpc,best_abs_mine_over_tobias_minus1_pct,best_mine_over_tobias_minus1_pct,mine_Omega_m,mine_z_eq,mine_Neff,tobias_Neff
0,DRMD,delta_Neff_drmd,0.0,1.0,0.0168,0.0168,0.05,0.0001,-0.0001,0.3158,3399.6589,3.0592,3.0592
1,DRMD,delta_Neff_drmd,0.2,1.0,0.4790,-0.4790,0.05,0.0000,0.0000,0.3158,3310.9247,3.2592,3.2592
2,DRMD,delta_Neff_drmd,0.4,1.0,0.4476,0.4476,0.05,0.0001,-0.0001,0.3158,3226.7048,3.4592,3.4592
3,DRMD,delta_Neff_drmd,0.6,1.0,0.5401,0.5401,0.05,0.0002,-0.0002,0.3158,3146.6634,3.6592,3.6592
4,DRMD,delta_Neff_drmd,0.8,1.0,0.0828,0.0828,0.05,0.0001,-0.0001,0.3158,3070.4968,3.8592,3.8592
5,DRMD,delta_Neff_drmd,1.0,1.0,0.2418,-0.2418,0.01,0.0001,-0.0001,0.3158,2997.9305,4.0592,4.0592


## Cell 6 — Scan `delta_Neff_drmd`

This cell varies the amount of extra dark radiation in the DRMD model while all other benchmark parameters are held fixed.

In [4]:
# Cell 7 — Run the delta_Neff_drmd scan

df_delta_neff = backend.run_parameter_scan(
    parameter_name="delta_Neff_drmd",
    scan_values=scan_ranges["delta_Neff_drmd"],
    benchmark=benchmark,
    k_values_1Mpc=k_values_1Mpc,
)

df_delta_neff

NameError: name 'backend' is not defined